In [ ]:
# 04 - Section 5 Statistical Analysis

This notebook reproduces the core Section 5 statistical-analysis pipeline used in the manuscript.

It reads the condition-family curve outputs generated by:

`03_condition_family_visualization.ipynb`

Expected input files:

`outputs/03_condition_family_visualization/family_*/family_curves.csv`

The notebook performs:

- SoC-domain masking and robustification
- Hampel outlier filtering
- optional running-median smoothing
- canonical voltage-shift contrasts
- practical-significance bucket assignment
- point-level factorial screening ANOVA with SoC as a covariate
- output generation for downstream calibration and evaluation notebooks

Important scope note: point-level ANOVA is used only as a dense screening and variance-decomposition summary. The dependence-reduced band-level HC3 analysis and aggressive 2C simple-effect summaries should be added as separate cells if not already generated by another notebook.

In [ ]:
# ============================================================
# Setup and configuration
# ============================================================

import os
import glob
import json
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd

import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore")

# Input generated by 03_condition_family_visualization.ipynb
IN_GLOB = "outputs/03_condition_family_visualization/family_*/family_curves.csv"

# Output directory for Section 5
OUT_DIR = "outputs/04_section5_statistical_analysis"
os.makedirs(OUT_DIR, exist_ok=True)

SOC_BINS = [0, 20, 60, 90, 100]
SOC_LABELS = ["0-20", "20-60", "60-90", "90-100"]

# Canonical inferential bands used in the manuscript
USE_SOC_BANDS = ["0-20", "20-60", "60-90"]

TEMP_LEVELS = [10, 25, 45, 60]

# Column convention inherited from the processed files:
#   Rate        -> charge cut-off current level: C/5 or C/40
#   Charge_Rate -> discharge-rate/rate-history level: 0.7C, 1C, or 2C
RATE_LEVELS = ["C/40", "C/5"]
CHARGE_LEVELS = ["0.7C", "1C", "2C"]

TOP_MASK_FRAC = 0.05

HAMPEL_K_SIGMA = 3.0
HAMPEL_WIN = 31

DO_SMOOTH_MEDIAN = True
SMOOTH_WIN = 7

MONOTONIZE_SOC = False

THRESH_SMALL = 0.05
THRESH_MEDIUM = 0.10
THRESH_LARGE = 0.15

RNG = np.random.default_rng(42)

print("Input glob:")
print(IN_GLOB)
print("\nOutput directory:")
print(OUT_DIR)

In [ ]:
# ============================================================
# Condition map: Condition_ID -> physical factor tuple
# ============================================================
# Tuple format:
#   Condition_ID: (Temperature, charge cut-off current, discharge-rate/rate-history)

COND_MAP = {
    1:  (10, "C/5",  "0.7C"),  2:  (10, "C/40", "0.7C"),
    3:  (10, "C/5",  "1C"),    4:  (10, "C/40", "1C"),
    5:  (10, "C/5",  "2C"),    6:  (10, "C/40", "2C"),
    7:  (25, "C/5",  "0.7C"),  8:  (25, "C/40", "0.7C"),
    9:  (25, "C/5",  "1C"),    10: (25, "C/40", "1C"),
    11: (25, "C/5",  "2C"),    12: (25, "C/40", "2C"),
    13: (45, "C/5",  "0.7C"),  14: (45, "C/40", "0.7C"),
    15: (45, "C/5",  "1C"),    16: (45, "C/40", "1C"),
    17: (45, "C/5",  "2C"),    18: (45, "C/40", "2C"),
    19: (60, "C/5",  "0.7C"),  20: (60, "C/40", "0.7C"),
    21: (60, "C/5",  "1C"),    22: (60, "C/40", "1C"),
    23: (60, "C/5",  "2C"),    24: (60, "C/40", "2C"),
}

condition_map_df = pd.DataFrame(
    [
        {
            "Condition_ID": cid,
            "Temperature": vals[0],
            "Rate": vals[1],
            "Charge_Rate": vals[2],
        }
        for cid, vals in COND_MAP.items()
    ]
)

display(condition_map_df)

In [ ]:
# ============================================================
# Safe numeric helpers
# ============================================================

def nanmedian_safe(a):
    a = np.asarray(a)
    if a.size == 0 or not np.isfinite(a).any():
        return np.nan
    return np.nanmedian(a)


def mad_safe(a, med=None):
    a = np.asarray(a)
    if a.size == 0 or not np.isfinite(a).any():
        return np.nan

    if med is None:
        med = nanmedian_safe(a)

    if not np.isfinite(med):
        return np.nan

    dev = np.abs(a - med)
    mad = nanmedian_safe(dev)

    return mad

In [ ]:
# ============================================================
# Input normalization helpers
# ============================================================

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    cols = list(df.columns)
    lower = {c.lower(): c for c in cols}

    def pick(candidates=None, contains=None):
        if candidates:
            for cand in candidates:
                key = cand.lower()
                if key in lower:
                    return lower[key]

        if contains:
            for c in cols:
                if contains.lower() in c.lower():
                    return c

        return None

    colmap = {}

    soc_col = pick(
        candidates=["SoC", "soc", "state_of_charge", "soc_%"],
        contains="soc"
    )
    if soc_col:
        colmap[soc_col] = "SoC"

    temp_col = pick(
        candidates=["Temperature", "Temp", "temperature_c", "ambient_temperature", "T"],
        contains="temp"
    )
    if temp_col:
        colmap[temp_col] = "Temperature"

    rate_col = pick(
        candidates=["Rate", "Discharge_Rate", "Test_Rate", "Discharge"],
        contains="rate"
    )
    if rate_col:
        colmap[rate_col] = "Rate"

    chg_col = pick(
        candidates=["Charge_Rate", "ChargeRate", "Charge-Rate", "CR", "Charge"],
        contains="charge"
    )
    if chg_col:
        colmap[chg_col] = "Charge_Rate"

    cond_col = pick(
        candidates=["Condition_ID", "ConditionId", "CondID", "Condition"],
        contains="condition"
    )
    if cond_col:
        colmap[cond_col] = "Condition_ID"

    if "V_median" in df.columns:
        colmap["V_median"] = "Voltage"
    else:
        if ("V_q25" in df.columns) and ("V_q75" in df.columns):
            df["Voltage"] = (
                pd.to_numeric(df["V_q25"], errors="coerce")
                + pd.to_numeric(df["V_q75"], errors="coerce")
            ) / 2.0
        else:
            volt_col = pick(
                candidates=["Voltage", "VoltageV", "V", "Cell_Voltage", "Voltage (V)", "volt"],
                contains="volt"
            )
            if volt_col:
                colmap[volt_col] = "Voltage"

    df2 = df.rename(columns=colmap)

    if "SoC" in df2.columns:
        df2["SoC"] = pd.to_numeric(df2["SoC"], errors="coerce")

    if "Voltage" in df2.columns:
        df2["Voltage"] = pd.to_numeric(df2["Voltage"], errors="coerce")

    return df2


def coerce_and_impute_from_condition(df: pd.DataFrame) -> pd.DataFrame:
    if "Condition_ID" in df.columns:
        df["Condition_ID"] = pd.to_numeric(
            df["Condition_ID"],
            errors="coerce"
        ).astype("Int64")

    required_factor_cols = ["Temperature", "Rate", "Charge_Rate"]

    for c in required_factor_cols:
        if c not in df.columns:
            df[c] = np.nan

    if "Condition_ID" in df.columns:
        mask_missing = (
            df["Temperature"].isna()
            | df["Rate"].isna()
            | df["Charge_Rate"].isna()
        )

        if mask_missing.any():
            def fill_row(row):
                cid = row.get("Condition_ID", pd.NA)

                try:
                    cid = int(cid)
                except Exception:
                    return row

                if cid in COND_MAP:
                    T, R, CR = COND_MAP[cid]

                    if pd.isna(row["Temperature"]):
                        row["Temperature"] = T

                    if pd.isna(row["Rate"]):
                        row["Rate"] = R

                    if pd.isna(row["Charge_Rate"]):
                        row["Charge_Rate"] = CR

                return row

            df = df.apply(fill_row, axis=1)

    if "Temperature" in df.columns:
        df["Temperature"] = pd.to_numeric(df["Temperature"], errors="coerce")

    return df

In [ ]:
# ============================================================
# Load and concatenate family-curve CSV files
# ============================================================

def load_concat(in_glob=IN_GLOB) -> pd.DataFrame:
    files = sorted(glob.glob(in_glob))

    if not files:
        raise FileNotFoundError(f"No files found for input glob: {in_glob}")

    dfs = []

    for fp in files:
        if fp.lower().endswith(".csv"):
            df = pd.read_csv(fp)
        else:
            df = pd.read_excel(fp)

        df["__source_file__"] = os.path.basename(fp)
        df["__source_path__"] = fp

        dfs.append(df)

    raw = pd.concat(dfs, ignore_index=True)

    df = normalize_columns(raw.copy())
    df = coerce_and_impute_from_condition(df)

    if "quantile" in df.columns:
        med_mask = df["quantile"].astype(str).str.lower().eq("median")
        if med_mask.any():
            df = df[med_mask].copy()

    required = ["SoC", "Voltage", "Temperature", "Rate", "Charge_Rate"]

    for c in required:
        if c in df.columns and df[c].dtype == object:
            df[c] = df[c].replace(
                {"nan": np.nan, "NaN": np.nan, "None": np.nan}
            )

    df = df.dropna(subset=required)

    if df.empty:
        diag_path = os.path.join(OUT_DIR, "load_diag.txt")
        with open(diag_path, "w", encoding="utf-8") as f:
            f.write("Data became empty after dropna.\n")
            f.write("Raw columns:\n")
            f.write(str(list(raw.columns)) + "\n")

        return df

    df["Temperature"] = pd.to_numeric(
        df["Temperature"],
        errors="coerce"
    ).astype(int)

    df["Rate"] = pd.Categorical(
        df["Rate"],
        categories=RATE_LEVELS,
        ordered=True
    )

    df["Charge_Rate"] = pd.Categorical(
        df["Charge_Rate"],
        categories=CHARGE_LEVELS,
        ordered=True
    )

    df["Temperature"] = pd.Categorical(
        df["Temperature"],
        categories=TEMP_LEVELS,
        ordered=True
    )

    df["SoC_band"] = pd.cut(
        df["SoC"].clip(0, 100),
        bins=SOC_BINS,
        labels=SOC_LABELS,
        include_lowest=True
    )

    return df


df_raw = load_concat()

print("Loaded family-curve data:")
print(f"Rows: {len(df_raw):,}")
print(f"Covered temperatures: {df_raw['Temperature'].dropna().unique()}")
print(f"Covered charge cut-off levels: {df_raw['Rate'].dropna().unique()}")
print(f"Covered discharge-rate/rate-history levels: {df_raw['Charge_Rate'].dropna().unique()}")

display(df_raw.head())

In [ ]:
# ============================================================
# Masking and robustification
# ============================================================

def mask_top_end(df: pd.DataFrame, top_frac=TOP_MASK_FRAC) -> pd.Series:
    threshold = 100.0 * (1.0 - float(top_frac))
    return df["SoC"] >= threshold


def hampel_mask_1d(y: np.ndarray, k=HAMPEL_K_SIGMA, win=HAMPEL_WIN) -> np.ndarray:
    if win % 2 == 0:
        win += 1

    y = np.asarray(y, dtype=float)
    n = len(y)

    if n == 0:
        return np.zeros(0, dtype=bool)

    half = win // 2
    mask = np.zeros(n, dtype=bool)

    for i in range(n):
        lo = max(0, i - half)
        hi = min(n, i + half + 1)

        window = y[lo:hi]
        med = nanmedian_safe(window)
        mad = mad_safe(window, med)

        if not np.isfinite(med) or not np.isfinite(mad):
            mask[i] = False
            continue

        scale = 1.4826 * mad if mad > 0 else np.nanstd(window)

        if not np.isfinite(scale) or scale == 0:
            mask[i] = False
            continue

        mask[i] = abs(y[i] - med) > k * scale

    return mask


def apply_masks_and_smooth(df: pd.DataFrame):
    d = df.copy().reset_index(drop=True)

    d["mask_top"] = mask_top_end(d)
    d["mask_hampel"] = False

    group_cols = ["Temperature", "Rate", "Charge_Rate"]
    reports = []

    def smooth_median(arr, win=SMOOTH_WIN):
        if win % 2 == 0:
            win += 1

        arr = np.asarray(arr, dtype=float)
        n = len(arr)

        if n == 0 or win <= 1:
            return arr

        half = win // 2
        out = arr.copy()

        for i in range(n):
            lo = max(0, i - half)
            hi = min(n, i + half + 1)

            window = arr[lo:hi]
            med = nanmedian_safe(window)

            if np.isfinite(med):
                out[i] = med

        return out

    d = d.sort_values(group_cols + ["SoC"]).reset_index(drop=True)

    for key, g in d.groupby(group_cols, sort=False, dropna=False):
        idx = g.index.to_numpy()

        hmask = hampel_mask_1d(
            g["Voltage"].to_numpy(),
            k=HAMPEL_K_SIGMA,
            win=HAMPEL_WIN
        )

        d.loc[idx, "mask_hampel"] = hmask

        v = g["Voltage"].to_numpy().astype(float)
        v_clean = v.copy()

        v_clean[hmask | d.loc[idx, "mask_top"].to_numpy()] = np.nan

        if DO_SMOOTH_MEDIAN:
            keep = ~np.isnan(v_clean)
            if keep.any():
                v_clean[keep] = smooth_median(v_clean[keep], win=SMOOTH_WIN)

        d.loc[idx, "Voltage_clean"] = v_clean

        reports.append({
            "Temperature": key[0],
            "Rate": key[1],
            "Charge_Rate": key[2],
            "n_total": int(len(g)),
            "n_mask_top": int(d.loc[idx, "mask_top"].sum()),
            "n_mask_hampel": int(np.sum(hmask)),
        })

    mask_report = pd.DataFrame.from_records(reports)

    if mask_report.empty:
        mask_report = pd.DataFrame(
            columns=[
                "Temperature", "Rate", "Charge_Rate",
                "n_total", "n_mask_top", "n_mask_hampel", "mask_ratio_%"
            ]
        )
    else:
        denom = mask_report["n_total"].replace(0, np.nan)
        ratio = (
            100.0
            * (
                mask_report["n_mask_top"].fillna(0)
                + mask_report["n_mask_hampel"].fillna(0)
            )
            / denom
        )
        mask_report["mask_ratio_%"] = ratio.fillna(0.0)

    d["SoC_band"] = pd.cut(
        d["SoC"].clip(0, 100),
        bins=SOC_BINS,
        labels=SOC_LABELS,
        include_lowest=True
    )

    return d, mask_report


clean, mask_report = apply_masks_and_smooth(df_raw)

clean_path = os.path.join(OUT_DIR, "tidy_clean.csv")
mask_report_path = os.path.join(OUT_DIR, "mask_report_by_group.csv")

clean.to_csv(clean_path, index=False)
mask_report.to_csv(mask_report_path, index=False)

print("Masking and robustification completed.")
print(f"Clean rows: {len(clean):,}")
print(f"Saved: {clean_path}")
print(f"Saved: {mask_report_path}")

display(mask_report.head())

In [ ]:
# ============================================================
# Effect-size and practical-bucket helpers
# ============================================================

@dataclass
class DiffSummary:
    contrast: str
    baseline: str
    levelset: dict
    soc_band: str
    n_baseline: int
    n_contrast: int
    mean_diff: float
    ci_low: float
    ci_high: float
    cliffs_delta: float
    glass_delta: float
    practical_bucket: str


def practical_bucket(delta_abs: float) -> str:
    if np.isnan(delta_abs):
        return "NA"
    if delta_abs >= THRESH_LARGE:
        return "large"
    if delta_abs >= THRESH_MEDIUM:
        return "moderate"
    if delta_abs >= THRESH_SMALL:
        return "small"
    return "negligible"


def bootstrap_ci(diff_samples: np.ndarray, alpha=0.05):
    if diff_samples.size == 0:
        return np.nan, np.nan

    lo, hi = np.quantile(diff_samples, [alpha / 2, 1 - alpha / 2])

    return float(lo), float(hi)


def cliffs_delta(x, y):
    x = np.asarray(x)
    y = np.asarray(y)

    nx, ny = len(x), len(y)

    if nx == 0 or ny == 0:
        return np.nan

    diffs = np.subtract.outer(y, x)

    return float((np.sum(diffs > 0) - np.sum(diffs < 0)) / (nx * ny))


def glass_delta(x_baseline, y_contrast):
    xb = np.asarray(x_baseline)
    yc = np.asarray(y_contrast)

    if xb.size < 2 or yc.size < 1:
        return np.nan

    s = np.std(xb, ddof=1)

    if s == 0:
        return np.nan

    return float((np.mean(yc) - np.mean(xb)) / s)

In [ ]:
# ============================================================
# Delta-V contrast table
# ============================================================

def delta_table(
    df: pd.DataFrame,
    factor: str,
    baseline,
    contrast,
    fixed: dict,
    use_bands=USE_SOC_BANDS,
    n_boot=1000,
    use_clean_voltage=True
) -> pd.DataFrame:

    needed = {"Rate", "Charge_Rate", "Temperature", "SoC_band"}

    if not needed.issubset(df.columns):
        return pd.DataFrame()

    vcol = "Voltage_clean" if use_clean_voltage and "Voltage_clean" in df.columns else "Voltage"

    sub = df.copy()

    for key, value in fixed.items():
        sub = sub[sub[key] == value]

    if (
        sub.empty
        or sub[sub[factor] == baseline].empty
        or sub[sub[factor] == contrast].empty
    ):
        return pd.DataFrame.from_records([
            {
                "contrast": str(contrast),
                "baseline": str(baseline),
                "levelset": fixed,
                "soc_band": band,
                "n_baseline": 0,
                "n_contrast": 0,
                "mean_diff": np.nan,
                "ci_low": np.nan,
                "ci_high": np.nan,
                "cliffs_delta": np.nan,
                "glass_delta": np.nan,
                "practical_bucket": "NA",
            }
            for band in use_bands
        ])

    records = []

    for band in use_bands:
        sb = sub[sub["SoC_band"] == band]

        xb = sb[sb[factor] == baseline][vcol].dropna().to_numpy()
        yc = sb[sb[factor] == contrast][vcol].dropna().to_numpy()

        if xb.size == 0 or yc.size == 0:
            md = np.nan
            ci = (np.nan, np.nan)
            cd = np.nan
            gd = np.nan
            bucket = "NA"

        else:
            md = float(np.mean(yc) - np.mean(xb))

            diffs = []

            for _ in range(n_boot):
                b = RNG.choice(xb, size=xb.size, replace=True)
                c = RNG.choice(yc, size=yc.size, replace=True)
                diffs.append(np.mean(c) - np.mean(b))

            ci = bootstrap_ci(np.asarray(diffs))
            cd = cliffs_delta(xb, yc)
            gd = glass_delta(xb, yc)
            bucket = practical_bucket(abs(md))

        rec = DiffSummary(
            contrast=str(contrast),
            baseline=str(baseline),
            levelset=fixed,
            soc_band=band,
            n_baseline=int(xb.size),
            n_contrast=int(yc.size),
            mean_diff=md,
            ci_low=ci[0],
            ci_high=ci[1],
            cliffs_delta=cd,
            glass_delta=gd,
            practical_bucket=bucket,
        ).__dict__

        records.append(rec)

    return pd.DataFrame.from_records(records)

In [ ]:
# ============================================================
# Canonical Delta-V contrasts
# ============================================================

# Charge cut-off current contrast:
# C/40 - C/5 at T = 25 °C and discharge-rate/rate-history level = 1C
delta_cutoff_T25_DR1C = delta_table(
    clean,
    factor="Rate",
    baseline="C/5",
    contrast="C/40",
    fixed={"Temperature": 25, "Charge_Rate": "1C"}
)

# Mild discharge-rate/rate-history contrast:
# 0.7C - 1C at T = 25 °C and charge cut-off current = C/40
delta_ratehist_T25_R_C40 = delta_table(
    clean,
    factor="Charge_Rate",
    baseline="1C",
    contrast="0.7C",
    fixed={"Temperature": 25, "Rate": "C/40"}
)

# Thermal contrast:
# 60 °C - 25 °C at charge cut-off current = C/40
# and discharge-rate/rate-history level = 1C
delta_temp_R_C40_DR1C = delta_table(
    clean,
    factor="Temperature",
    baseline=25,
    contrast=60,
    fixed={"Rate": "C/40", "Charge_Rate": "1C"}
)

paths = {
    "delta_cutoff_T25_DR1C.csv": delta_cutoff_T25_DR1C,
    "delta_ratehist_T25_R_C40.csv": delta_ratehist_T25_R_C40,
    "delta_temp_R_C40_DR1C.csv": delta_temp_R_C40_DR1C,
}

for filename, df_out in paths.items():
    out_path = os.path.join(OUT_DIR, filename)
    df_out.to_csv(out_path, index=False)
    print("Saved:", out_path)

print("\nCharge cut-off current contrast:")
display(delta_cutoff_T25_DR1C)

print("\nMild discharge-rate/rate-history contrast:")
display(delta_ratehist_T25_R_C40)

print("\nThermal contrast:")
display(delta_temp_R_C40_DR1C)

In [ ]:
# ============================================================
# Canonical contrast grids for heat-map and bucket summaries
# ============================================================

rows = []

for T in TEMP_LEVELS:
    dT = delta_table(
        clean,
        factor="Rate",
        baseline="C/5",
        contrast="C/40",
        fixed={"Temperature": T, "Charge_Rate": "1C"}
    )
    dT["Temperature"] = T
    rows.append(dT)

cutoff_grid = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
cutoff_grid.rename(columns={"soc_band": "SoC_band"}, inplace=True)
cutoff_grid.to_csv(
    os.path.join(OUT_DIR, "heatmap_delta_cutoff_grid.csv"),
    index=False
)

rows = []

for T in TEMP_LEVELS:
    dT = delta_table(
        clean,
        factor="Charge_Rate",
        baseline="1C",
        contrast="0.7C",
        fixed={"Temperature": T, "Rate": "C/40"}
    )
    dT["Temperature"] = T
    rows.append(dT)

ratehist_grid = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
ratehist_grid.rename(columns={"soc_band": "SoC_band"}, inplace=True)
ratehist_grid.to_csv(
    os.path.join(OUT_DIR, "heatmap_delta_ratehist_grid.csv"),
    index=False
)

rows = []

for dr in CHARGE_LEVELS:
    ddr = delta_table(
        clean,
        factor="Temperature",
        baseline=25,
        contrast=60,
        fixed={"Rate": "C/40", "Charge_Rate": dr}
    )
    ddr["Charge_Rate"] = dr
    rows.append(ddr)

temp_grid = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
temp_grid.rename(columns={"soc_band": "SoC_band"}, inplace=True)
temp_grid.to_csv(
    os.path.join(OUT_DIR, "heatmap_delta_temp_grid.csv"),
    index=False
)

print("Saved canonical contrast grids:")
print(" - heatmap_delta_cutoff_grid.csv")
print(" - heatmap_delta_ratehist_grid.csv")
print(" - heatmap_delta_temp_grid.csv")

In [ ]:
# ============================================================
# Three-way point-level ANOVA with SoC as covariate
# ============================================================

def three_way_anova(
    df: pd.DataFrame,
    use_bands=USE_SOC_BANDS,
    use_clean_voltage=True
) -> pd.DataFrame:

    if df is None or df.empty:
        out = pd.DataFrame()
        out.to_csv(os.path.join(OUT_DIR, "anova_TxIcutxDrate_with_SoC.csv"))
        with open(os.path.join(OUT_DIR, "anova_model_summary.txt"), "w", encoding="utf-8") as f:
            f.write("Empty dataframe; ANOVA skipped.\n")
        return out

    vcol = "Voltage_clean" if use_clean_voltage and "Voltage_clean" in df.columns else "Voltage"

    d = df[df["SoC_band"].isin(use_bands)].copy()

    if d.empty or d[vcol].dropna().size < 8:
        out = pd.DataFrame()
        out.to_csv(os.path.join(OUT_DIR, "anova_TxIcutxDrate_with_SoC.csv"))
        with open(os.path.join(OUT_DIR, "anova_model_summary.txt"), "w", encoding="utf-8") as f:
            f.write("Insufficient rows for ANOVA after filtering.\n")
        return out

    d["SoC_cont"] = d["SoC"] / 100.0

    for c in ["Temperature", "Rate", "Charge_Rate"]:
        d[c] = d[c].astype("category")

    try:
        model = smf.ols(
            f"{vcol} ~ C(Temperature)*C(Rate)*C(Charge_Rate) + SoC_cont",
            data=d
        ).fit()

        aov = anova_lm(model, typ=2)

        aov_path = os.path.join(OUT_DIR, "anova_TxIcutxDrate_with_SoC.csv")
        summary_path = os.path.join(OUT_DIR, "anova_model_summary.txt")

        aov.to_csv(aov_path)

        with open(summary_path, "w", encoding="utf-8") as f:
            f.write(model.summary().as_text())

        print("ANOVA completed.")
        print("Saved:", aov_path)
        print("Saved:", summary_path)

        return aov

    except Exception as e:
        summary_path = os.path.join(OUT_DIR, "anova_model_summary.txt")
        with open(summary_path, "w", encoding="utf-8") as f:
            f.write(f"ANOVA failed: {e}\n")

        pd.DataFrame().to_csv(
            os.path.join(OUT_DIR, "anova_TxIcutxDrate_with_SoC.csv")
        )

        print("ANOVA failed:", e)

        return pd.DataFrame()


anova_table = three_way_anova(clean)

display(anova_table)

In [ ]:
# ============================================================
# Run summary
# ============================================================

summary = {
    "n_rows_clean": int(len(clean)),
    "soc_bands": USE_SOC_BANDS,
    "temp_levels": TEMP_LEVELS,
    "charge_cutoff_levels_Rate_column": RATE_LEVELS,
    "discharge_rate_history_levels_Charge_Rate_column": CHARGE_LEVELS,
    "top_mask_frac": TOP_MASK_FRAC,
    "hampel_k_sigma": HAMPEL_K_SIGMA,
    "hampel_win": HAMPEL_WIN,
    "smooth_median": DO_SMOOTH_MEDIAN,
    "smooth_win": SMOOTH_WIN,
    "practical_thresholds_V": {
        "small": THRESH_SMALL,
        "moderate": THRESH_MEDIUM,
        "large": THRESH_LARGE,
    },
}

summary_path = os.path.join(OUT_DIR, "run_summary.json")

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("Section 5 core statistical-analysis pipeline completed.")
print("Outputs directory:", OUT_DIR)
print("Saved:", summary_path)

In [ ]:
## Expected outputs

Main output directory:

`outputs/04_section5_statistical_analysis/`

Core outputs generated by this notebook:

- `tidy_clean.csv`
- `mask_report_by_group.csv`
- `delta_cutoff_T25_DR1C.csv`
- `delta_ratehist_T25_R_C40.csv`
- `delta_temp_R_C40_DR1C.csv`
- `heatmap_delta_cutoff_grid.csv`
- `heatmap_delta_ratehist_grid.csv`
- `heatmap_delta_temp_grid.csv`
- `anova_TxIcutxDrate_with_SoC.csv`
- `anova_model_summary.txt`
- `run_summary.json`

These outputs provide the core Section 5 statistical-analysis inputs for downstream sparse gated calibration and LUT construction.

Additional outputs required for the full manuscript tables, such as aggressive 2C simple effects, dependence-reduced band-level HC3 analysis, canonical bucket counts, and robustness mini-checks, should be added as additional cells or generated by companion scripts if not already available.